
# How a Force Field Works: An Interactive Tour of Bonded and Non-Bonded Terms
### Stretching, bending, torsion, out-of-plane bending, van der Waals, and Coulomb

This is the **first** notebook of the course. Before we build any real molecule (ethanol, in a later notebook) or run any dynamics (also a later notebook), we need to understand the basic building block underneath *every* classical force field, including GAFF: a **potential energy function**.

A force field's total potential energy is always written as a sum of simple, independent terms:

$$ U_{\text{total}} = \underbrace{\sum_{\text{bonds}} U_{\text{stretch}}}_{\text{2 atoms}} + \underbrace{\sum_{\text{angles}} U_{\text{bend}}}_{\text{3 atoms}} + \underbrace{\sum_{\text{dihedrals}} U_{\text{torsion}}}_{\text{4 atoms}} + \underbrace{\sum_{\text{impropers}} U_{\text{out-of-plane}}}_{\text{4 atoms}} + \underbrace{\sum_{\text{pairs}} \left( U_{\text{vdW}} + U_{\text{Coulomb}} \right)}_{\text{2 non-bonded atoms}} $$

The first four terms are **bonded** (they only apply to atoms connected through 1, 2, or 3 bonds); the last two are **non-bonded** (they apply to essentially every other pair of atoms in the system). In this notebook you will:

- see the mathematical shape of each term,
- **change its parameters live** with sliders and watch the curve reshape itself,
- **change the actual geometry** (bond length, angle, dihedral angle...) with another slider and watch both the 3D picture of the atoms *and* the energy update together.

All six functional forms below are exactly the ones used by OpenMM (we checked each formula numerically against OpenMM's own force classes while preparing this notebook) — this is precisely the machinery the ethanol/GAFF notebook uses "under the hood".

**A note on interactivity.** This notebook uses `ipywidgets` sliders. If the sliders don't render as sliders (you just see text like `interactive(children=(...))`), run `pip install ipywidgets` (or, in a conda environment, `conda install -c conda-forge ipywidgets`) and restart Jupyter.



## 0. Setup


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (enables 3D projection)
from ipywidgets import interact, FloatSlider, IntSlider

plt.rcParams["figure.dpi"] = 100

# OpenMM's Coulomb constant, ke = 1/(4 pi epsilon0), in units of kJ*nm/(mol*e^2).
# (Verified directly against openmm.NonbondedForce while preparing this notebook.)
COULOMB_KE = 138.935456



## 1. Bond stretching (2 atoms) — a harmonic spring

The simplest bonded term treats a chemical bond like a tiny spring: it has a natural, most comfortable length $r_0$, and it costs energy to stretch or compress it away from $r_0$. OpenMM's `HarmonicBondForce` uses:

$$ U_{\text{stretch}}(r) = \tfrac{1}{2} k_b (r - r_0)^2 $$

- $k_b$ (in kJ/mol/nm$^2$) controls how *stiff* the spring is — a stiffer bond (larger $k_b$) makes the energy rise more sharply away from $r_0$.
- $r_0$ (in nm) is the equilibrium bond length — the bottom of the well.
- $r$ is the actual, instantaneous distance between the two atoms.

Try it: move **k_b** and **r_0** to see the shape and position of the well change; move **r** to see where a real bond at that distance would sit on the curve (and how much energy that costs).


In [7]:
def bond_energy(r, k_b, r0):
    return 0.5 * k_b * (r - r0) ** 2

def plot_bond(k_b, r0, r):
    plt.close("all")
    fig = plt.figure(figsize=(9, 3.8))

    # --- left: energy curve ---
    ax1 = fig.add_subplot(1, 2, 1)
    r_axis = np.linspace(0.0, 0.30, 300)
    ax1.plot(r_axis, bond_energy(r_axis, k_b, r0), color="tab:blue")
    E_now = bond_energy(r, k_b, r0)
    ax1.plot([r], [E_now], "o", color="crimson", ms=8)
    ax1.axvline(r0, color="gray", ls=":", lw=1)
    ax1.set_xlabel("bond length r (nm)")
    ax1.set_ylim(-3., 50)
    ax1.set_ylabel("U (kJ/mol)")
    ax1.set_title(f"U(r);  current: r={r:.3f} nm,  E={E_now:.1f} kJ/mol")

    # --- right: the two atoms ---
    ax2 = fig.add_subplot(1, 2, 2)
    ax2.plot([-r/2, r/2], [0, 0], "-", color="black", lw=2, zorder=1)
    ax2.scatter([-r/2, r/2], [0, 0], s=800, color=["tab:orange", "tab:cyan"], zorder=2, edgecolor="k")
    ax2.set_xlim(-0.20, 0.20); ax2.set_ylim(-0.1, 0.1)
    ax2.set_aspect("equal"); ax2.axis("off")
    ax2.set_title("the two atoms")

    plt.tight_layout()
    plt.show()

interact(
    plot_bond,
    k_b=FloatSlider(min=1000, max=10000, step=50, value=2000, description="k_b", continuous_update=False),
    r0=FloatSlider(min=0.09, max=0.17, step=0.005, value=0.15, description="r0 (nm)", continuous_update=False),
    r=FloatSlider(min=0.0, max=0.3, step=0.005, value=0.15, description="r (nm)", continuous_update=False),
);


interactive(children=(FloatSlider(value=2000.0, continuous_update=False, description='k_b', max=10000.0, min=1…


**Something to notice:** because $U(r)$ keeps rising forever as $r$ grows, a harmonic bond can never actually *break*, no matter how far you stretch it. Real chemical bonds obviously can break — this is one of the well-known limitations of a simple harmonic force field (more advanced/reactive potentials, like Morse potentials, are built specifically to allow dissociation, at the cost of extra parameters and complexity).



## 2. Angle bending (3 atoms) — a harmonic hinge

Three atoms in a row, $A-B-C$, define an angle at the central atom $B$. Just like a bond has a preferred length, this angle has a preferred value $\theta_0$, and bending it away from $\theta_0$ costs energy. OpenMM's `HarmonicAngleForce` uses the same mathematical shape as the bond term, just with an angle instead of a distance:

$$ U_{\text{bend}}(\theta) = \tfrac{1}{2} k_a (\theta - \theta_0)^2 $$

Here $\theta$ is in **radians** inside the formula (that's what makes $k_a$'s units, kJ/mol/rad$^2$, work out correctly), even though we show the sliders in the more intuitive **degrees**.


In [3]:
def angle_energy_deg(theta_deg, k_a, theta0_deg):
    theta = np.radians(theta_deg)
    theta0 = np.radians(theta0_deg)
    return 0.5 * k_a * (theta - theta0) ** 2

def angle_geometry(theta_deg, bond_len=0.14):
    theta = np.radians(theta_deg)
    B = np.array([0.0, 0.0])
    A = np.array([bond_len, 0.0])
    C = np.array([bond_len * np.cos(theta), bond_len * np.sin(theta)])
    return A, B, C

def plot_angle(k_a, theta0_deg, theta_deg):
    plt.close("all")
    fig = plt.figure(figsize=(9, 3.8))

    ax1 = fig.add_subplot(1, 2, 1)
    theta_axis = np.linspace(40, 180, 300)
    ax1.plot(theta_axis, angle_energy_deg(theta_axis, k_a, theta0_deg), color="tab:blue")
    E_now = angle_energy_deg(theta_deg, k_a, theta0_deg)
    ax1.plot([theta_deg], [E_now], "o", color="crimson", ms=8)
    ax1.axvline(theta0_deg, color="gray", ls=":", lw=1)
    ax1.set_ylim(-1, 100)
    ax1.set_xlabel("angle theta (degrees)")
    ax1.set_ylabel("U (kJ/mol)")
    ax1.set_title(f"U(theta);  current: theta={theta_deg:.1f} deg,  E={E_now:.1f} kJ/mol")

    ax2 = fig.add_subplot(1, 2, 2)
    A, B, C = angle_geometry(theta_deg)
    xs, ys = [A[0], B[0], C[0]], [A[1], B[1], C[1]]
    ax2.plot(xs, ys, "-", color="black", lw=2, zorder=1)
    ax2.scatter(xs, ys, s=[500, 800, 500], color=["tab:orange", "tab:green", "tab:cyan"], zorder=2, edgecolor="k")
    ax2.set_xlim(-0.1, 0.20); ax2.set_ylim(-0.05, 0.20)
    ax2.set_aspect("equal"); ax2.axis("off")
    ax2.set_title("A - B (central) - C")

    plt.tight_layout()
    plt.show()

interact(
    plot_angle,
    k_a=FloatSlider(min=100, max=500, step=20, value=300, description="k_a", continuous_update=False),
    theta0_deg=FloatSlider(min=90, max=180, step=1, value=109.5, description="theta0 (deg)", continuous_update=False),
    theta_deg=FloatSlider(min=45, max=120, step=5, value=109.5, description="theta (deg)", continuous_update=False),
);


interactive(children=(FloatSlider(value=300.0, continuous_update=False, description='k_a', max=500.0, min=100.…


## 3. Torsion / proper dihedral (4 atoms) — a periodic "twist"

Now take a chain of four atoms, $1-2-3-4$. Fix the two end bonds and the two bond angles, and let the whole $1$-$2$ end **rotate** around the central $2-3$ bond, relative to the $3$-$4$ end. The angle of that rotation is the **dihedral (torsion) angle** $\phi$.

Unlike bonds and angles, torsions are usually **periodic**: rotating a full $360°$ brings you back to where you started, and there are often several energetically equivalent orientations (e.g., a bond that can freely rotate has multiple "comfortable" positions per full turn, not just one). OpenMM's `PeriodicTorsionForce` captures this with a cosine:

$$ U_{\text{torsion}}(\phi) = k_t\,\big[\,1 + \cos(n\phi - \gamma)\,\big] $$

- $k_t$ sets the height of the energy barrier that must be crossed to rotate the bond.
- $n$ (the **multiplicity**, a small positive integer) sets how many energy minima occur per full $360°$ turn (n=3 is typical for a bond attached to an sp$^3$ carbon, like a C-C single bond in ethane; n=2 is typical around a planar/sp$^2$ center; n=1 describes a single preferred orientation).
- $\gamma$ (the **phase**) shifts where those minima sit.


In [5]:
def torsion_energy_deg(phi_deg, k_t, n, gamma_deg):
    phi = np.radians(phi_deg)
    gamma = np.radians(gamma_deg)
    return k_t * (1 + np.cos(n * phi - gamma))

def dihedral_angle(p1, p2, p3, p4):
    """Standard dihedral angle (radians) for the chain p1-p2-p3-p4."""
    b1, b2, b3 = p2 - p1, p3 - p2, p4 - p3
    n1, n2 = np.cross(b1, b2), np.cross(b2, b3)
    m1 = np.cross(n1, b2 / np.linalg.norm(b2))
    return np.arctan2(np.dot(m1, n2), np.dot(n1, n2))

def place_atom4(p1, p2, p3, bond_length, angle_234, dihedral_1234):
    """NeRF placement of atom 4 given atoms 1,2,3 and internal coordinates for atom 4."""
    bc = (p3 - p2) / np.linalg.norm(p3 - p2)
    nvec = np.cross(p2 - p1, bc); nvec /= np.linalg.norm(nvec)
    mvec = np.cross(nvec, bc)
    d2 = np.array([
        -bond_length * np.cos(angle_234),
         bond_length * np.sin(angle_234) * np.cos(-dihedral_1234),
         bond_length * np.sin(angle_234) * np.sin(-dihedral_1234),
    ])
    M = np.column_stack([bc, mvec, nvec])
    return p3 + M @ d2

# Fixed "backbone" geometry: bond lengths and bond angles held constant,
# only the dihedral angle phi (the rotation around the 2-3 bond) will vary.
R_BOND, THETA_BEND = 0.15, np.radians(109.5)
P2 = np.array([0.0, 0.0, 0.0])
P3 = np.array([R_BOND, 0.0, 0.0])
P1 = P2 + R_BOND * np.array([np.cos(np.pi - THETA_BEND), np.sin(np.pi - THETA_BEND), 0.0])

def torsion_geometry(phi_deg):
    phi = np.radians(phi_deg)
    p4 = place_atom4(P1, P2, P3, R_BOND, THETA_BEND, phi)
    return P1, P2, P3, p4

def plot_torsion(k_t, n, gamma_deg, phi_deg):
    plt.close("all")
    fig = plt.figure(figsize=(10, 4.2))

    ax1 = fig.add_subplot(1, 2, 1)
    phi_axis = np.linspace(-180, 180, 300)
    ax1.plot(phi_axis, torsion_energy_deg(phi_axis, k_t, n, gamma_deg), color="tab:blue")
    E_now = torsion_energy_deg(phi_deg, k_t, n, gamma_deg)
    ax1.plot([phi_deg], [E_now], "o", color="crimson", ms=8)
    ax1.set_ylim(-1, 41)
    ax1.set_xlabel("dihedral angle phi (degrees)")
    ax1.set_ylabel("U (kJ/mol)")
    ax1.set_title(f"U(phi);  current: phi={phi_deg:.0f} deg,  E={E_now:.2f} kJ/mol")

    ax2 = fig.add_subplot(1, 2, 2, projection="3d")
    p1, p2, p3, p4 = torsion_geometry(phi_deg)
    pts = np.array([p1, p2, p3, p4])
    ax2.plot(pts[:, 0], pts[:, 1], pts[:, 2], "-", color="black", lw=2)
    colors = ["tab:orange", "tab:green", "tab:green", "tab:cyan"]
    ax2.scatter(pts[:, 0], pts[:, 1], pts[:, 2], s=250, c=colors, edgecolor="k", depthshade=False)
    ax2.set_title("atoms 1-2-3-4 (rotating around the 2-3 bond)")
    ax2.set_box_aspect([1, 1, 1])
    for axis in (ax2.set_xlim, ax2.set_ylim, ax2.set_zlim):
        axis(-0.2, 0.2)
    ax2.set_axis_off()

    plt.tight_layout()
    plt.show()

interact(
    plot_torsion,
    k_t=FloatSlider(min=0, max=20, step=0.5, value=5, description="k_t", continuous_update=False),
    n=IntSlider(min=1, max=6, step=1, value=3, description="n", continuous_update=False),
    gamma_deg=FloatSlider(min=0, max=180, step=180, value=0, description="gamma (deg)", continuous_update=False),
    phi_deg=FloatSlider(min=-180, max=180, step=5, value=60, description="phi (deg)", continuous_update=False),
);


interactive(children=(FloatSlider(value=5.0, continuous_update=False, description='k_t', max=20.0, step=0.5), …


Set $n=3$ and watch the curve: it now has **three** equally-spaced minima between $-180°$ and $180°$ — exactly the three staggered ("gauche"/"anti") orientations you may recall from organic chemistry for rotation around a C-C single bond. Try $n=1$ or $n=2$ and see how the number of minima changes to match.



## 4. Out-of-plane bending / improper torsion (4 atoms) — keeping things flat

Some centers in a molecule are supposed to stay (roughly) **flat** — the classic example is a carbonyl carbon (C=O), which is $sp^2$-hybridized and wants its three substituents (the oxygen and two others) arranged in a plane around it. If one substituent gets pushed out of that plane, an **out-of-plane** (a.k.a. "improper torsion") term penalizes it.

Different force fields implement this idea slightly differently (some, like GAFF/AMBER, reuse the same periodic torsion formula from Section 3 on a specially chosen quadruplet of atoms; others, like CHARMM, use a dedicated harmonic term). Here we use the more intuitive harmonic version, in the same spirit as the bond and angle terms above:

$$ U_{\text{oop}}(\chi) = \tfrac{1}{2} k_{\chi}\, \chi^2 $$

where $\chi$ (chi) is the angle between the bond to the "test" atom $D$ and the plane defined by the other three atoms ($A$, $B$, $C$). When $\chi = 0$, everything is perfectly flat and the energy is zero, exactly like a bond/angle term centered at its equilibrium value — except here the "equilibrium" is always planarity.


In [5]:
def oop_energy_deg(chi_deg, k_chi):
    chi = np.radians(chi_deg)
    return 0.5 * k_chi * chi ** 2

def oop_geometry(chi_deg, bond_len=0.14):
    chi = np.radians(chi_deg)
    C = np.array([0.0, 0.0, 0.0])
    A = bond_len * np.array([np.cos(np.radians(120)), np.sin(np.radians(120)), 0.0])
    B = bond_len * np.array([np.cos(np.radians(240)), np.sin(np.radians(240)), 0.0])
    D = bond_len * np.array([np.cos(chi), 0.0, np.sin(chi)])  # D lifts out of the A-B-C plane by chi
    return A, B, C, D

def plot_oop(k_chi, chi_deg):
    plt.close("all")
    fig = plt.figure(figsize=(10, 4.2))

    ax1 = fig.add_subplot(1, 2, 1)
    chi_axis = np.linspace(-90, 90, 300)
    ax1.plot(chi_axis, oop_energy_deg(chi_axis, k_chi), color="tab:blue")
    E_now = oop_energy_deg(chi_deg, k_chi)
    ax1.plot([chi_deg], [E_now], "o", color="crimson", ms=8)
    ax1.axvline(0, color="gray", ls=":", lw=1, label="planar (chi=0)")
    ax1.set_ylim(-1, 250)
    ax1.set_xlabel("out-of-plane angle chi (degrees)")
    ax1.set_ylabel("U (kJ/mol)")
    ax1.set_title(f"U(chi);  current: chi={chi_deg:.0f} deg,  E={E_now:.1f} kJ/mol")
    ax1.legend(fontsize=8)

    ax2 = fig.add_subplot(1, 2, 2, projection="3d")
    A, B, C, D = oop_geometry(chi_deg)
    # shade the reference plane (z=0) spanned by A, B, C for visual reference
    plane_pts = np.array([A, B, C])
    ax2.plot_trisurf(plane_pts[:, 0], plane_pts[:, 1], plane_pts[:, 2], color="lightgray", alpha=0.4)
    for p, label in [(A, "A"), (B, "B")]:
        ax2.plot([C[0], p[0]], [C[1], p[1]], [C[2], p[2]], "-", color="black", lw=2)
    ax2.plot([C[0], D[0]], [C[1], D[1]], [C[2], D[2]], "-", color="crimson", lw=2)
    pts = np.array([A, B, C, D])
    colors = ["tab:orange", "tab:cyan", "tab:green", "crimson"]
    ax2.scatter(pts[:, 0], pts[:, 1], pts[:, 2], s=250, c=colors, edgecolor="k", depthshade=False)
    ax2.set_title("C (central) bonded to A, B, D;\nplane = A-B-C, D moves out of it")
    ax2.set_box_aspect([1, 1, 1])
    for axis in (ax2.set_xlim, ax2.set_ylim, ax2.set_zlim):
        axis(-0.16, 0.16)
    ax2.set_axis_off()

    plt.tight_layout()
    plt.show()

interact(
    plot_oop,
    k_chi=FloatSlider(min=0, max=200, step=10, value=100, description="k_chi", continuous_update=False),
    chi_deg=FloatSlider(min=-89, max=89, step=1, value=0, description="chi (deg)", continuous_update=False),
);


interactive(children=(FloatSlider(value=100.0, continuous_update=False, description='k_chi', max=200.0, step=1…


## 5. Non-bonded interactions: van der Waals (Lennard-Jones)

So far, every term needed atoms connected by bonds. But **any** two atoms that are *not* directly bonded (and not involved in a shared angle) still interact — through van der Waals forces and, if they carry partial charges, through electrostatics. There is no fixed "equilibrium distance" here in the same sense as a bond; instead, the interaction just depends on however far apart the two atoms happen to be at any moment.

We met the Lennard-Jones potential already in the previous notebook:

$$ U_{\text{vdW}}(r) = 4\varepsilon \left[ \left(\frac{\sigma}{r}\right)^{12} - \left(\frac{\sigma}{r}\right)^{6} \right] $$


In [2]:
def lj_energy(r, epsilon, sigma):
    sr6 = (sigma / r) ** 6
    return 4 * epsilon * (sr6 ** 2 - sr6)

def plot_vdw(epsilon, sigma, r):
    plt.close("all")
    fig = plt.figure(figsize=(9, 3.8))

    ax1 = fig.add_subplot(1, 2, 1)
    r_axis = np.linspace(0.2, 1.0, 300)
    ax1.plot(r_axis, lj_energy(r_axis, epsilon, sigma), color="tab:blue")
    E_now = lj_energy(r, epsilon, sigma)
    ax1.plot([r], [E_now], "o", color="crimson", ms=8)
    ax1.axhline(0, color="gray", lw=0.6)
    ax1.set_ylim(-2.5, 3)
    ax1.set_xlabel("distance r (nm)")
    ax1.set_ylabel("U (kJ/mol)")
    ax1.set_title(f"U_vdW(r);  current: r={r:.3f} nm,  E={E_now:.3f} kJ/mol")

    ax2 = fig.add_subplot(1, 2, 2)
    ax2.plot([-r/2, r/2], [0, 0], "--", color="gray", lw=1.5, zorder=1)  # dashed: NOT a chemical bond
    ax2.scatter([-r/2, r/2], [0, 0], s=800, color=["tab:orange", "tab:cyan"], zorder=2, edgecolor="k")
    ax2.set_xlim(-0.55, 0.55); ax2.set_ylim(-0.2, 0.2)
    ax2.set_aspect("equal"); ax2.axis("off")
    ax2.set_title("two non-bonded atoms")

    plt.tight_layout()
    plt.show()

interact(
    plot_vdw,
    epsilon=FloatSlider(min=0.1, max=2.0, step=0.05, value=0.8, description="epsilon", continuous_update=False),
    sigma=FloatSlider(min=0.25, max=0.45, step=0.01, value=0.34, description="sigma (nm)", continuous_update=False),
    r=FloatSlider(min=0.20, max=0.90, step=0.01, value=0.40, description="r (nm)", continuous_update=False),
);


interactive(children=(FloatSlider(value=0.8, continuous_update=False, description='epsilon', max=2.0, min=0.1,…


Notice the shape is the *same* curve from the LJ-argon notebook — but now you can directly feel what $\varepsilon$ and $\sigma$ each control: $\varepsilon$ is purely the **depth** of the attractive well (how "sticky" the pair is), while $\sigma$ shifts the **whole curve sideways** (how "big" the atoms effectively are, i.e., how close they can get before repulsion kicks in).



## 6. Non-bonded interactions: Coulombic (electrostatics)

If the two atoms carry partial charges $q_1$ and $q_2$ (in units of the elementary charge $e$ — recall the Gasteiger charges from the ethanol notebook), they also interact through Coulomb's law:

$$ U_{\text{Coulomb}}(r) = k_e \frac{q_1 q_2}{r}, \qquad k_e = 138.935\ \text{kJ}\cdot\text{nm}/(\text{mol}\cdot e^2) $$

Opposite charges ($q_1 q_2 < 0$) attract (negative energy, more favorable at short range); like charges ($q_1 q_2 > 0$) repel (positive energy). Unlike the Lennard-Jones term, there is no built-in repulsive wall here — Coulomb's law alone would let two opposite charges collapse to $r=0$ with infinite (negative) energy; in a real molecule, the Lennard-Jones repulsion between the same two atoms always stops that from happening (see Section 7).


In [3]:
def coulomb_energy(r, q1, q2):
    return COULOMB_KE * q1 * q2 / r

def plot_coulomb(q1, q2, r):
    plt.close("all")
    fig = plt.figure(figsize=(9, 3.8))

    ax1 = fig.add_subplot(1, 2, 1)
    r_axis = np.linspace(0.1, 1.0, 300)
    ax1.plot(r_axis, coulomb_energy(r_axis, q1, q2), color="tab:blue")
    E_now = coulomb_energy(r, q1, q2)
    ax1.plot([r], [E_now], "o", color="crimson", ms=8)
    ax1.axhline(0, color="gray", lw=0.6)
    ax1.set_xlabel("distance r (nm)")
    ax1.set_ylabel("U (kJ/mol)")
    ax1.set_title(f"U_Coulomb(r);  current: r={r:.3f} nm,  E={E_now:.1f} kJ/mol")

    ax2 = fig.add_subplot(1, 2, 2)
    ax2.plot([-r/2, r/2], [0, 0], "--", color="gray", lw=1.5, zorder=1)
    colors = ["tab:red" if q1 > 0 else "tab:blue", "tab:red" if q2 > 0 else "tab:blue"]
    ax2.scatter([-r/2, r/2], [0, 0], s=800, color=colors, zorder=2, edgecolor="k")
    ax2.text(-r/2, 0.05, f"q1={q1:+.2f}e", ha="center")
    ax2.text(r/2, 0.05, f"q2={q2:+.2f}e", ha="center")
    ax2.set_xlim(-0.55, 0.55); ax2.set_ylim(-0.15, 0.15)
    ax2.set_aspect("equal"); ax2.axis("off")
    ax2.set_title("two charged, non-bonded atoms")

    plt.tight_layout()
    plt.show()

interact(
    plot_coulomb,
    q1=FloatSlider(min=-1.0, max=1.0, step=0.1, value=0.4, description="q1 (e)", continuous_update=False),
    q2=FloatSlider(min=-1.0, max=1.0, step=0.1, value=-0.4, description="q2 (e)", continuous_update=False),
    r=FloatSlider(min=0.10, max=0.90, step=0.01, value=0.35, description="r (nm)", continuous_update=False),
);


interactive(children=(FloatSlider(value=0.4, continuous_update=False, description='q1 (e)', max=1.0, min=-1.0)…


## 7. Putting vdW and Coulomb together

In a real simulation, every non-bonded pair feels **both** terms simultaneously — the total non-bonded energy is just their sum, $U_{\text{vdW}} + U_{\text{Coulomb}}$. Play with all five parameters below and watch how the two contributions trade off: Coulomb tends to dominate at longer range (it decays only as $1/r$), while the Lennard-Jones repulsion always wins at very short range (it decays/grows as $1/r^{12}$, much faster than anything else), preventing the electrostatic collapse mentioned above.



## 8. Summary and what comes next

| Term | Atoms involved | Functional form | What it penalizes |
|---|---|---|---|
| Stretching | 2 (bonded) | $\tfrac12 k_b (r-r_0)^2$ | deviation from equilibrium bond length |
| Bending | 3 (bonded) | $\tfrac12 k_a (\theta-\theta_0)^2$ | deviation from equilibrium bond angle |
| Torsion | 4 (bonded) | $k_t[1+\cos(n\phi-\gamma)]$ | rotation away from preferred dihedral orientations |
| Out-of-plane | 4 (bonded) | $\tfrac12 k_\chi \chi^2$ | loss of planarity at an $sp^2$-like center |
| van der Waals | 2 (non-bonded) | $4\varepsilon[(\sigma/r)^{12}-(\sigma/r)^6]$ | atoms getting too close (repulsion) / missing out on weak attraction |
| Coulomb | 2 (non-bonded, charged) | $k_e q_1 q_2 / r$ | unfavorable arrangement of partial charges |

Every one of these six terms will reappear, unchanged, inside the **ethanol/GAFF notebook**: there, `HarmonicBondForce`, `HarmonicAngleForce`, `PeriodicTorsionForce`, and `NonbondedForce` (vdW + Coulomb together) are exactly the OpenMM classes that implement the formulas above, just with real, atom-type-specific parameters ($k_b$, $r_0$, $k_a$, $\theta_0$, ...) looked up from the GAFF parameter file instead of chosen by you on a slider. And the **MD fundamentals notebook** showed you how a simulation actually moves atoms through the landscape defined by the *sum* of all these terms, using the Velocity Verlet algorithm.

## Discussion questions

1. In Section 1, what happens to the energy if you set $k_b$ very high and then look at how far $r$ can realistically wander during a simulation running at room temperature? What does this tell you about why bond vibrations need a *short* integration timestep (as we saw in the MD fundamentals notebook)?
2. For the torsion term (Section 3), what is the physical difference between changing $n$ and changing $\gamma$? Can you set up two different $(k_t, n, \gamma)$ combinations that happen to give the *same* energy at $\phi=0°$ but different energies everywhere else?
3. In Section 4, is $\chi$ allowed to be negative? What would that correspond to physically?
4. In Section 7, find parameter values where the *total* non-bonded curve has **two** local minima instead of one. Under what physical circumstances might that matter for a real molecule?
5. Ethanol's hydroxyl oxygen carries a substantial negative partial charge (from the ethanol notebook: about $-0.40\,e$) and its hydroxyl hydrogen a large positive one (about $+0.21\,e$). Using the sliders in Section 6, explore roughly how much energy is associated with an O$\cdots$H arrangement at a typical hydrogen-bonding distance ($r\approx 0.19$ nm) versus a normal non-bonded distance ($r\approx 0.35$ nm). Does the size of that difference surprise you?
